In [ ]:
! pip install crewai crewai-tools python-dotenv

In [5]:
import os
from dotenv import load_dotenv

load_dotenv()
print(os.getenv("GROQ_API_KEY") is not None)

True


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

print(os.getenv("SERPER_API_KEY") is not None)

In [2]:
import crewai
import litellm
print(crewai.__version__)

1.15.41


In [8]:
import os
from dotenv import load_dotenv
from crewai import LLM

load_dotenv(override=True)

llm = LLM(
    model="groq/openai/gpt-oss-120b",  # Best quality for writing & tool use
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0.7
)

print(llm.call("What is AI?"))

**Artificial Intelligence (AI)** is a branch of computer science focused on creating systems that can perform tasks that normally require human intelligence. These tasks include learning, reasoning, problem‑solving, perception, language understanding, and even creativity. In practice, AI systems aim to mimic—or augment—cognitive functions such as:

| Core Capability | What It Means | Typical Techniques |
|-----------------|---------------|--------------------|
| **Learning** | Improving performance from data or experience | Machine learning (supervised, unsupervised, reinforcement), deep neural networks |
| **Reasoning & Planning** | Drawing logical conclusions, making decisions, and planning sequences of actions | Symbolic AI, rule‑based systems, probabilistic graphical models, search algorithms |
| **Perception** | Interpreting sensory input (vision, sound, touch) | Computer vision, speech recognition, sensor fusion |
| **Natural Language Processing (NLP)** | Understanding and genera

In [1]:
import os
from dotenv import load_dotenv

from crewai import Agent, Task, Crew, Process, LLM
from crewai_tools import SerperDevTool

# Fix for Groq + CrewAI cache_breakpoint issue
import crewai.llms.cache as crewai_cache
crewai_cache.mark_cache_breakpoint = lambda msg: msg

# Reload .env with override=True so updated keys take effect immediately
load_dotenv(override=True)

# Limit Serper to 3 results to keep prompt tokens under Groq's 8,000 TPM cap
search_tool = SerperDevTool(n_results=3)

# Split agents across separate Groq models so each gets its own TPM bucket
researcher_llm = LLM(
    model="groq/openai/gpt-oss-20b",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0.3,
    max_tokens=1024
)

analyst_llm = LLM(
    model="groq/qwen/qwen3.8-27b",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0.3,
    max_tokens=1024
)

writer_llm = LLM(
    model="groq/openai/gpt-oss-120b",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0.5,
    max_tokens=2048
)

researcher = Agent(
    role="Senior Research Agent",
    goal="Research the topic and collect accurate, useful information.",
    backstory=(
        "You are an expert researcher. You collect key facts, trends, "
        "examples, and references from reliable sources."
    ),
    tools=[search_tool],
    llm=researcher_llm,
    max_iter=3,  # Prevents excessive search loops that burn TPM quota
    verbose=True
)

analyst = Agent(
    role="Content Analyst Agent",
    goal="Analyze the research and extract the most important insights.",
    backstory=(
        "You are an expert content strategist. You convert raw research "
        "into structured insights, key points, and a clear content outline."
    ),
    llm=analyst_llm,
    max_iter=2,
    verbose=True
)

writer = Agent(
    role="Blog Writer Agent",
    goal="Write a beginner-friendly blog post from the analyzed insights.",
    backstory=(
        "You are a clear and practical blog writer. You explain complex "
        "topics in simple language with examples."
    ),
    llm=writer_llm,
    max_iter=2,
    verbose=True
)

editor = Agent(
    role="Editor Agent",
    goal="Improve clarity, grammar, flow, and final presentation.",
    backstory=(
        "You are an experienced editor. You polish content and make it "
        "easy to read, structured, and professional."
    ),
    llm=writer_llm,
    max_iter=2,
    verbose=True
)


def build_blog_crew(topic: str) -> Crew:
    research_task = Task(
        description=(
            f"Research the topic: {topic}. "
            "Collect important facts, use cases, beginner-friendly examples, "
            "and useful references. Keep the summary concise."
        ),
        expected_output=(
            "A concise, structured research summary with key facts, examples, "
            "use cases, and references."
        ),
        agent=researcher
    )

    analysis_task = Task(
        description=(
            f"Analyze the research for the topic: {topic}. "
            "Create a clear blog outline with key sections, examples, and main points."
        ),
        expected_output=(
            "A structured blog outline with introduction, main sections, "
            "practical examples, and conclusion."
        ),
        agent=analyst,
        context=[research_task]
    )

    writing_task = Task(
        description=(
            f"Write a complete beginner-friendly blog post on: {topic}. "
            "Use the structured outline provided. Keep it simple, clear, "
            "and practical."
        ),
        expected_output=(
            "A complete blog post in Markdown with title, introduction, "
            "headings, examples, and conclusion."
        ),
        agent=writer,
        context=[analysis_task]  # Avoids duplicating raw research tokens
    )

    editing_task = Task(
        description=(
            "Review and improve the blog post. Fix grammar, improve flow, "
            "make it more readable, and add a strong final summary."
        ),
        expected_output=(
            "Final polished blog post in Markdown ready to publish."
        ),
        agent=editor,
        context=[writing_task]
    )

    crew = Crew(
        agents=[researcher, analyst, writer, editor],
        tasks=[research_task, analysis_task, writing_task, editing_task],
        process=Process.sequential,
        max_rpm=2,  # Paces requests to stay within Groq's 8,000 TPM limit
        verbose=True
    )

    return crew


# Run in Jupyter Notebook cell
topic = input("Enter blog topic: ")
crew = build_blog_crew(topic)

result = await crew.kickoff_async()

print("\n\n================ FINAL BLOG ================\n")
print(result.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 1cc5f061-ddb6-4ff1-9370-bedf9e1fa5a6                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Research the topic: Harness Engineering. Collect important facts, use cases, beginner-friendly           │
│  examples, and useful references. Keep the summary concise.                                                     │
│  ID: 218f067b-e894-4ee2-ab97-d07288fb0d20                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Research Agent                                                                                   │
│                                                                                                                 │
│  Task: Research the topic: Harness Engineering. Collect important facts, use cases, beginner-friendly           │
│  examples, and useful references. Keep the summary concise.                                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_the_internet_with_serper                                                                          │
│  Args: {'search_query': 'Harness Engineering definition and use cases'}                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────── ✅ Tool Execution Completed (#1) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: search_the_internet_with_serper                                                                          │
│  Output: {'searchParameters': {'q': 'Harness Engineering definition and use cases', 'type': 'search', 'num':    │
│  3, 'engine': 'google'}, 'organic': [{'title': 'Harness engineering for coding agent users', 'link':            │
│  'https://martinfowler.com/articles/harness-engineering.html', 'snippet': 'The term harness has emerged as a    │
│  shorthand to mean everything in an AI agent except the model itself - Agent = Model + Harness. That is a very  │
│  wide definition,', 'position': 1}, {'title': 'The Anatomy of an Agent Harness', 'link':                        │
│  'https://www.langchain.com/blog/the-anatomy-of-an-agent-harness', 'snippet': 'Harness engineering is how we    │
│  build systems around models to turn them into work engines. The model contains the intelligence and the        │
│  harness makes that ...', 'position': 2}, {'title': 'Harness engineering: leveraging Codex in an agent-first    │
│  world', 'link': 'https://openai.com/index/harness-engineering/', 'snippet': 'Agents use our standard           │
│  development tools directly. They pull review feedback, respond inline, push updates, and often squash and      │
│  merge ...', 'position': 3}], 'peopleAlsoAsk': [{'question': 'What is harness engineering in simple terms?',    │
│  'snippet': '', 'title': '', 'link': ''}, {'question': 'Can you provide an example of harness engineering?',    │
│  'snippet': '', 'title': '', 'link': ''}, {'question': 'What does a harness engineer do?', 'snippet': '',       │
│  'title': '', 'link': ''}], 'relatedSearches': [{'query': 'Harness engineering definition and use cases         │
│  github'}, {'query': 'What is harness engineering in AI'}, {'query': 'Harness engineering Anthropic'}],         │
│  'credits': 1}                                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#2) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_the_internet_with_serper                                                                          │
│  Args: {'search_query': 'Harness engineering AI agent harness examples'}                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (20 outputs left out)

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Editor Agent                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Harness Engineering – Building the Body for Your AI Brain                                                    │
│                                                                                                                 │
│  *Turning a powerful language model into a reliable, safe, and useful AI agent.*                                │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## Introduction: The Model Is Not the Agent                                                                    │
│                                                                                                                 │
│  When you hear “use an LLM (large language model)”, it’s easy to assume the job is done. In reality, a raw      │
│  model is **just a brain in a jar** – it can reason and generate text, but it has no hands to act, no memory    │
│  to recall, and no immune system to keep it safe.                                                               │
│                                                                                                                 │
│  > **Agent = Model + Harness**                                                                                  │
│                                                                                                                 │
│  | Part      | What it is                              | What it does                                           │
│  |                                                                                                              │
│  |-----------|------------------------------------------|-----------------------------------------------------  │
│  -----------------------|                                                                                       │
│  | **Model** | The core LLM (GPT‑4, Claude, Llama, …)  | Performs reasoning, language understanding, and        │
│  generation.               |                                                                                    │
│  | **Harness**| The surrounding infrastructure – the “body” around the brain | Supplies context, memory,        │
│  tools, safety checks, observability, and deployment. |                                                         │
│                                                                                                                 │
│  Without a well‑engineered harness, agents are brittle, unsafe, and unable to handle multi‑step tasks.          │
│  **Harness Engineering** is the discipline of building that reliable “body” so the model can interact with the  │
│  real world.                                                                                                    │
│                                                                                                                 │
│  > *Martin Fowler* describes **“Harness Engineering for Coding Agent Users”** as the practice of                │
│  **systematically wiring up models with the plumbing they need to be production‑ready**. His definition gives   │
│  us a solid footing for the discussion that follows.   

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Review and improve the blog post. Fix grammar, improve flow, make it more readable, and add a strong     │
│  final summary.                                                                                                 │
│  Agent: Editor Agent                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 1cc5f061-ddb6-4ff1-9370-bedf9e1fa5a6                                                                       │
│  Final Output: # Harness Engineering – Building the Body for Your AI Brain                                      │
│                                                                                                                 │
│  *Turning a powerful language model into a reliable, safe, and useful AI agent.*                                │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## Introduction: The Model Is Not the Agent                                                                    │
│                                                                                                                 │
│  When you hear “use an LLM (large language model)”, it’s easy to assume the job is done. In reality, a raw      │
│  model is **just a brain in a jar** – it can reason and generate text, but it has no hands to act, no memory    │
│  to recall, and no immune system to keep it safe.                                                               │
│                                                                                                                 │
│  > **Agent = Model + Harness**                                                                                  │
│                                                                                                                 │
│  | Part      | What it is                              | What it does                                           │
│  |                                                                                                              │
│  |-----------|------------------------------------------|-----------------------------------------------------  │
│  -----------------------|                                                                                       │
│  | **Model** | The core LLM (GPT‑4, Claude, Llama, …)  | Performs reasoning, language understanding, and        │
│  generation.               |                                                                                    │
│  | **Harness**| The surrounding infrastructure – the “body” around the brain | Supplies context, memory,        │
│  tools, safety checks, observability, and deployment. |                                                         │
│                                                                                                                 │
│  Without a well‑engineered harness, agents are brittle, unsafe, and unable to handle multi‑step tasks.          │
│  **Harness Engineering** is the discipline of building that reliable “body” so the model can interact with the  │
│  real world.                                                                                                    │
│                                                                                                                 │
│  > *Martin Fowler* describes **“Harness Engineering for Coding Agent Users”** as the practice of                │
│  **systematically wiring up models with the plumbing they need to be production‑ready**. His definition gives   │
│  us a solid footing for the discussion that follows.  



================ FINAL BLOG ================

# Harness Engineering – Building the Body for Your AI Brain  

*Turning a powerful language model into a reliable, safe, and useful AI agent.*

---

## Introduction: The Model Is Not the Agent  

When you hear “use an LLM (large language model)”, it’s easy to assume the job is done. In reality, a raw model is **just a brain in a jar** – it can reason and generate text, but it has no hands to act, no memory to recall, and no immune system to keep it safe.

> **Agent = Model + Harness**

| Part      | What it is                              | What it does                                                               |
|-----------|------------------------------------------|----------------------------------------------------------------------------|
| **Model** | The core LLM (GPT‑4, Claude, Llama, …)  | Performs reasoning, language understanding, and generation.               |
| **Harness**| The surrounding infrastructure – the “body” aro